# ECFP4 Encoder Sweep Results

plots the CSVs `scripts/ECFP4-encoder-sweep.py` writes to `outputs/encoders/ECFP4/sweep/`, so run the sweep first. this only reads them, so it takes a few seconds

configs are ranked by mean val RMSE across seeds, error bars are std across seeds. test only shows up for the finalists, since that's the only place it gets used

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)

SWEEP = Path("../outputs/encoders/ECFP4/sweep")
stages = {p.stem: pd.read_csv(p) for p in sorted(SWEEP.glob("*.csv"))}

CONFIG_KEYS = [
    "N_BITS", "HIDDEN_DIMS", "DROPOUT", "FINAL_ACTIVATION", "LR",
    "WEIGHT_DECAY", "BATCH_SIZE", "EPOCHS", "OPT", "MODEL", "ALPHA", "MAX_FEATURES",
]
SHORT = {
    "N_BITS": "bits", "HIDDEN_DIMS": "dims", "DROPOUT": "drop", "FINAL_ACTIVATION": "final_act",
    "LR": "lr", "WEIGHT_DECAY": "wd", "BATCH_SIZE": "batch",
    "EPOCHS": "epochs", "OPT": "opt", "ALPHA": "alpha", "MAX_FEATURES": "max_features",
}
METRICS = ["train_RMSE", "val_RMSE", "test_RMSE", "test_R2", "test_ROC-AUC", "best_epoch"]

{name: len(df) for name, df in stages.items()}

In [ ]:
# config keys that differ between rows
def varying(df):
    return [k for k in CONFIG_KEYS if k in df and df[k].nunique(dropna=False) > 1]


def with_labels(df):
    keys = varying(df)
    labels = df.groupby("cfg")[keys].first().apply(
        lambda r: ", ".join(
            str(v) if k == "MODEL" else f"{SHORT[k]}={v}" for k, v in r.items() if not pd.isna(v)
        ),
        axis=1,
    )
    return df.assign(config=df["cfg"].map(labels))


# one row per config: mean over seeds + the val spread
def summarize(df):
    g = (df if "config" in df else with_labels(df)).groupby("config")
    out = g[[m for m in METRICS if m in df]].mean()
    out["val_RMSE_std"] = g["val_RMSE"].std()
    out["seeds"] = g.size()
    return out.sort_values("val_RMSE")


def show(name, n=10):
    s = summarize(stages[name]).head(n)
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(s) + 1.2))
    ax.errorbar(s["val_RMSE"], range(len(s)), xerr=s["val_RMSE_std"], fmt="o", capsize=3)
    ax.set(yticks=range(len(s)), yticklabels=s.index, xlabel="val RMSE", title=f"{name} (top {len(s)})")
    ax.invert_yaxis()
    plt.show()
    return s.round(3)

## 1-regularization: regularization and width

mean val RMSE for each `DROPOUT` × `WEIGHT_DECAY` cell, one panel per `HIDDEN_DIMS`. if the best cells sit on an edge of the grid, the next stage should extend past that edge

In [ ]:
s1 = stages["1-regularization"]
dims = sorted(s1["HIDDEN_DIMS"].unique(), key=len, reverse=True)
grids = {
    d: s1[s1["HIDDEN_DIMS"] == d].pivot_table(
        index="DROPOUT", columns="WEIGHT_DECAY", values="val_RMSE", aggfunc="mean"
    )
    for d in dims
}
vmin = min(g.min().min() for g in grids.values())
vmax = max(g.max().max() for g in grids.values())

fig, axes = plt.subplots(1, len(dims), figsize=(4.5 * len(dims), 4), sharey=True, layout="constrained")
for ax, (d, grid) in zip(axes, grids.items()):
    im = ax.imshow(grid, cmap="viridis_r", vmin=vmin, vmax=vmax)
    for (r, c), v in np.ndenumerate(grid.to_numpy()):
        ax.text(c, r, f"{v:.3f}", ha="center", va="center", color="k" if im.norm(v) < 0.5 else "w")
    ax.set_xticks(range(grid.shape[1]), [f"{w:g}" for w in grid.columns])
    ax.set_yticks(range(grid.shape[0]), grid.index)
    ax.set(title=f"HIDDEN_DIMS={d}", xlabel="WEIGHT_DECAY")
axes[0].set_ylabel("DROPOUT")
fig.colorbar(im, ax=axes, label="mean val RMSE (lower is better)")
plt.show()

## 2-regularization-wider, 3-training-dynamics, 5-final-activation

each stage's top configs. overlapping error bars mean the gap is within seed noise

In [ ]:
show("2-regularization-wider")

In [ ]:
show("3-training-dynamics")

In [ ]:
show("5-final-activation")

## Finalists vs baselines

finalists from `4-holdout-seeds` and `6-final-activation-holdout-seeds`, plus the best ridge and random forest by val, all on seeds 3-7 (ridge is deterministic, so its seeds overlap). dots are single seeds, diamonds are mean ± std. pick by the val panel, test is for reporting

In [ ]:
final = pd.concat([stages["4-holdout-seeds"], stages["6-final-activation-holdout-seeds"]], ignore_index=True)

baselines = stages["7-baselines"]
val_means = baselines.groupby("cfg")["val_RMSE"].mean()
models = baselines.groupby("cfg")["MODEL"].first()
best = [val_means[models == m].idxmin() for m in models.unique()]

cmp = pd.concat([with_labels(final), with_labels(baselines[baselines["cfg"].isin(best)])])
order = cmp.groupby("config")["val_RMSE"].mean().sort_values().index

fig, axes = plt.subplots(1, 2, figsize=(15, 0.5 * len(order) + 1.5), sharey=True, layout="constrained")
for ax, metric in zip(axes, ["val_RMSE", "test_RMSE"]):
    for i, c in enumerate(order):
        v = cmp.loc[cmp["config"] == c, metric]
        ax.scatter(v, np.full(len(v), i), color="C0", alpha=0.5)
        ax.errorbar(v.mean(), i, xerr=v.std(), fmt="D", color="k", capsize=3)
    ax.set(xlabel=metric, yticks=range(len(order)), yticklabels=order)
axes[0].invert_yaxis()
axes[0].set_title("val (selection)")
axes[1].set_title("test (reporting only)")
plt.show()

summarize(cmp).loc[order].round(3)